# 🚗 CS Frotas — Pipeline End-to-End no BigQuery Studio (Colab Enterprise)

Este notebook executa o pipeline completo do **MVP CS Frotas - Nexus CA** diretamente no ambiente do **[BigQuery Studio / Notebooks no BigQuery](https://docs.cloud.google.com/bigquery/docs/create-notebooks?hl=pt)** (baseado no Colab Enterprise).

### 📋 Etapas do Pipeline:
1. **Ambiente & Conexão**: Inicialização dos clientes BigQuery e Cloud Storage com credenciais nativas da sessão GCP.
2. **Ingestão & Sanitização (GCS -> BigQuery)**: Leitura das planilhas brutas do Google Cloud Storage (`gs://...`), aplicação de regras de negócio e carga nas tabelas do dataset `cs_frotas_data`.
3. **Cruzamento Vetor x SAP**: Criação da View analítica `vw_cruzamento_vetor_sap` com identificação de divergências.
4. **De-Para Semântico com IA (BigQuery ML)**: Desduplicação de catálogos e correlação inteligente Vetor x SAP com Gemini.
5. **Auditoria de Sobrepreço e Cobrança de Avaria (CA)**: Inferência com `ML.GENERATE_TEXT` via Vertex AI (Gemini 3.7 Flash).

## 1. Configuração e Dependências

In [ ]:
# Instalação de pacotes auxiliares no runtime do BigQuery Notebook
!pip install --upgrade google-cloud-bigquery google-cloud-storage pandas openpyxl db-dtypes gcsfs -q

In [ ]:
import io
import os
import re
import pandas as pd
from google.cloud import bigquery, storage

# Defina seu ID de Projeto, Dataset e Bucket do GCS
PROJECT_ID = os.getenv('GCP_PROJECT_ID', 'your-gcp-project-id')
DATASET_ID = os.getenv('BQ_DATASET_ID', 'cs_frotas_data')
GCS_BUCKET = os.getenv('GCS_BUCKET', 'your-gcs-bucket-name')

# Inicializar Clientes GCP (autenticação herdada automaticamente no BigQuery Studio)
bq_client = bigquery.Client(project=PROJECT_ID)
storage_client = storage.Client(project=PROJECT_ID)

print(f'✅ Conectado ao BigQuery (Projeto: {PROJECT_ID}, Dataset: {DATASET_ID})')
print(f'🪣 Bucket de Origem GCS: gs://{GCS_BUCKET}')

## 2. Funções de Ingestão e Sanitização de Dados

In [ ]:
def get_gcs_or_local_file(uri_or_path: str):
    '''Lê arquivo de bucket GCS (gs://...) ou do filesystem local.'''
    if uri_or_path.startswith('gs://'):
        path_without_prefix = uri_or_path[5:]
        bucket_name, blob_name = path_without_prefix.split('/', 1)
        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(blob_name)
        print(f'📥 Baixando {uri_or_path}...')
        return io.BytesIO(blob.download_as_bytes())
    return uri_or_path

def sanitize_column_name(col, idx):
    '''Sanitiza nomes de colunas para conformidade com regras do BigQuery.'''
    if not col or str(col).strip() == '' or str(col) == 'None':
        return f'coluna_{idx}'
    s = str(col).strip()
    s = s.replace('ª', '').replace('º', '').replace('%', 'pct').replace('/', '_').replace('.', '')
    s = re.sub(r'[^\w\s]', '_', s)
    s = re.sub(r'\s+', '_', s).lower()
    if re.match(r'^\d', s):
        s = 'c_' + s
    return s

## 3. Carga das Bases do VETOR e SAP MB52 no BigQuery

In [ ]:
# URIs dos arquivos no Google Cloud Storage
vetor_uri = f'gs://{GCS_BUCKET}/Manutenção - Relatório de Item VETOR.xlsx'
sap_uri = f'gs://{GCS_BUCKET}/Manutenção - Relatório de Estoque SAP.xlsx'

# --- 1. CARGA DA BASE VETOR ---
print('📖 Lendo e processando VETOR...')
df_vetor = pd.read_excel(get_gcs_or_local_file(vetor_uri), sheet_name='Relatorio_de_Item', dtype=str)
df_vetor.columns = [sanitize_column_name(c, i) for i, c in enumerate(df_vetor.columns)]

for col in df_vetor.columns:
    if any(k in col for k in ['valor', 'preco', 'desconto', 'variacao', 'reducao']):
        df_vetor[col] = pd.to_numeric(df_vetor[col].astype(str).str.replace(',', '.'), errors='coerce')
    elif any(k in col for k in ['quantidade', 'idade', 'km', 'dia', 'mes', 'ano']):
        df_vetor[col] = pd.to_numeric(df_vetor[col], errors='coerce')

# Filtro de Ordens Reprovadas
status_col = [c for c in df_vetor.columns if 'status' in c]
if status_col:
    df_vetor = df_vetor[~df_vetor[status_col[0]].astype(str).str.upper().str.contains('REPROVAD', na=False)].copy()

t_vetor = f'{PROJECT_ID}.{DATASET_ID}.relatorio_item_vetor'
bq_client.load_table_from_dataframe(
    df_vetor, t_vetor, 
    job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE', autodetect=True)
).result()
print(f'✅ Tabela `{t_vetor}` criada com {len(df_vetor):,} registros!')

# --- 2. CARGA DA BASE SAP MB52 ---
print('📖 Lendo e processando SAP MB52...')
df_sap = pd.read_excel(get_gcs_or_local_file(sap_uri), sheet_name='MB52', dtype=str)
df_sap.columns = [sanitize_column_name(c, i) for i, c in enumerate(df_sap.columns)]
for col in df_sap.columns:
    if any(k in col for k in ['val', 'utilizacao', 'transito']):
        df_sap[col] = pd.to_numeric(df_sap[col].astype(str).str.replace(',', '.'), errors='coerce')

t_sap = f'{PROJECT_ID}.{DATASET_ID}.relatorio_estoque_sap_mb52'
bq_client.load_table_from_dataframe(
    df_sap, t_sap,
    job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE', autodetect=True)
).result()
print(f'✅ Tabela `{t_sap}` criada com {len(df_sap):,} registros!')

## 4. Criação da View Analítica de Cruzamento

In [ ]:
%%bigquery
-- Criação da View Consolidada de Divergências Vetor x SAP
CREATE OR REPLACE VIEW `cs_frotas_data.vw_cruzamento_vetor_sap` AS
SELECT
  v.`código_item_vetor` AS codigo_item_vetor,
  v.`código_item_sap` AS codigo_item_sap_ref,
  s.material AS codigo_material_sap,
  s.`texto_breve_de_material` AS descricao_sap,
  
  SAFE_CAST(v.quantidade AS NUMERIC) AS qtd_vetor,
  SAFE_CAST(s.`utilização_livre` AS NUMERIC) AS qtd_sap_livre,
  SAFE_CAST(v.valor_total AS NUMERIC) AS valor_total_vetor,
  SAFE_CAST(s.valutilizlivre AS NUMERIC) AS valor_total_sap,
  (SAFE_CAST(v.valor_total AS NUMERIC) - SAFE_CAST(s.valutilizlivre AS NUMERIC)) AS dif_valor,
  
  CASE 
    WHEN s.material IS NULL THEN 'Presente Apenas no Vetor'
    WHEN v.`código_item_vetor` IS NULL THEN 'Presente Apenas no SAP'
    WHEN ABS(SAFE_CAST(v.valor_total AS NUMERIC) - SAFE_CAST(s.valutilizlivre AS NUMERIC)) > 100 THEN 'Divergência Relevante'
    ELSE 'Consistente'
  END AS status_divergencia

FROM `cs_frotas_data.relatorio_item_vetor` v
FULL OUTER JOIN `cs_frotas_data.relatorio_estoque_sap_mb52` s
  ON v.`código_item_sap` = s.material;

## 5. Auditoria de Anomalias com Gemini 3.7 Flash no BigQuery ML

In [ ]:
%%bigquery
CREATE OR REPLACE MODEL `cs_frotas_data.gemini_flash_model`
REMOTE WITH CONNECTION DEFAULT
OPTIONS(endpoint = 'https://aiplatform.googleapis.com/v1/projects/dacommunitybr/locations/global/publishers/google/models/gemini-3.7-flash');

In [ ]:
%%bigquery df_anomalias
-- Consulta Generativa de Auditoria com Gemini 3.7 Flash
SELECT
  ml_generate_text_result AS analise_ia,
  codigo_item_vetor,
  valor_total_vetor,
  valor_total_sap,
  dif_valor
FROM
  ML.GENERATE_TEXT(
    MODEL `cs_frotas_data.gemini_flash_model`,
    (
      SELECT
        CONCAT(
          'Atue como um auditor de suprimentos e frotas. Analise a divergência:\n',
          'Item Vetor: ', IFNULL(descricao_sap, 'N/A'), ' | Valor Vetor: R$ ', CAST(IFNULL(valor_total_vetor, 0) AS STRING), '\n',
          'Item SAP: ', IFNULL(descricao_sap, 'N/A'), ' | Valor SAP: R$ ', CAST(IFNULL(valor_total_sap, 0) AS STRING), '\n',
          'Diferença: R$ ', CAST(IFNULL(dif_valor, 0) AS STRING)
        ) AS prompt,
        codigo_item_vetor,
        valor_total_vetor,
        valor_total_sap,
        dif_valor
      FROM `cs_frotas_data.vw_cruzamento_vetor_sap`
      WHERE status_divergencia = 'Divergência Relevante'
      LIMIT 5
    ),
    STRUCT(0.2 AS temperature, 300 AS max_output_tokens)
  );

In [ ]:
# Visualizar análises de IA geradas
display(df_anomalias)